<a href="https://colab.research.google.com/github/gggelme/geo-project/blob/main/src/notebooks/modeling/entrenamiento_2m_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrenamiento con el train completo (~1.93M imágenes) — Colab

Notebook delgado: todo el entrenamiento vive en `src/entrenamiento` (el mismo código de la prueba local de 200k).
Acá solo se prepara la máquina y se lanza `scripts/entrenar.py --config configs/completo_2m.json`.

**Para empezar o retomar: Ejecutar todo.** La corrida guarda todo en Drive (`carpeta_corrida`) y sigue sola desde
el último checkpoint. Si la sesión se corta, se reconecta y se vuelve a ejecutar todo: se vuelven a bajar los
parquets (el disco de la máquina se borra) y el entrenamiento retoma donde quedó.

**Dos corridas en la misma sesión:** cuando termina una, cambiar `CONFIG` en la celda de parámetros, ejecutarla
y ejecutar solo la celda de entrenamiento (sección 4): los datos ya están en el disco y no se vuelven a bajar.

**Para pausar a mano:** interrumpir la celda de entrenamiento (■). El script recibe la señal, guarda un checkpoint
y sale. `MAX_HORAS` hace lo mismo solo, antes de que Colab corte la sesión.

**Datos:** al empezar cada sesión se bajan los parquets de train y val y se **recomprimen a 384 px (JPEG calidad
90)** en el disco de la máquina: de ~181 GB a ~77 GB, para que entren en los ~121 GB de la L4. El modelo sigue
viendo 224 px con la misma augmentation: el RandomResizedCrop (60-100% del área) y el Resize de validación siempre
achican, y el recorte más chico de una imagen de 384 px (~297 px) también. Se conservan el orden de las filas y las
demás columnas, así que los splits y el sampler no cambian.

Por qué no se leen de HF de a uno durante el entrenamiento: cada ronda es balanceada por celda, así que toca casi
todos los parquets; leerlos de HF en cada ronda sería bajar ~180 GB por ronda.

In [1]:
# ---- Parámetros de la sesión ----
import os
RAMA = "main"
# Qué se entrena. La carpeta de la corrida en Drive lleva el nombre de la config.
#   configs/completo_2m.json        celdas K=70 + focal haversine + SAM + augmentation + snapshots (ya entrenado)
#   configs/baseline_celdas_2m.json celdas K=70, CE simple, sin augmentation, sin SAM, sin snapshots
#   configs/baseline_paises_2m.json 71 países (>= 1000 filas, como el de 20k), CE simple, sin augmentation/SAM/snapshots
CONFIG = "configs/baseline_paises_2m.json"
DRIVE = "/content/drive/MyDrive/geo-project"     # Carpeta de Drive del proyecto.
INSUMOS_DRIVE = f"{DRIVE}/insumos"                # manifest.parquet, splits.parquet, validation.parquet (subidos a mano).
CARPETA_CORRIDA = f"{DRIVE}/entrenamiento_2m/{os.path.splitext(os.path.basename(CONFIG))[0]}"
MAX_HORAS = 20.0    # Por sesión. Por debajo del límite de Colab (24 h en Pro) para que guarde y salga solo.
REPO = "/content/geo-project"
SHARDS = "/content/shards"                        # Disco local de la máquina (rápido, se borra al cerrar la sesión).

## 1. Máquina

In [2]:
import os, shutil, subprocess
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPUs:", os.cpu_count())
!free -g | head -2
total, usado, libre = shutil.disk_usage("/content")
# Si la sesión ya recomprimió parte de los datos (se relanzó sin reiniciar), eso cuenta como espacio disponible.
hechos = sum(e.stat().st_size for e in os.scandir(f"{SHARDS}/data") if e.is_file()) if os.path.isdir(f"{SHARDS}/data") else 0
print(f"Disco /content: {libre / 1e9:.0f} GB libres de {total / 1e9:.0f} GB ({hechos / 1e9:.0f} GB ya recomprimidos)")
assert libre + hechos > 100e9, "Hacen falta ~80 GB para los parquets recomprimidos de train y val (+ margen)."

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
CPUs: 12
               total        used        free      shared  buff/cache   available
Mem:              52           1          48           0           3          51
Disco /content: 121 GB libres de 121 GB (0 GB ya recomprimidos)


In [3]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(CARPETA_CORRIDA, exist_ok=True)
for nombre in ["manifest.parquet", "splits.parquet", "validation.parquet"]:
    assert os.path.exists(f"{INSUMOS_DRIVE}/{nombre}"), f"Falta subir {nombre} a {INSUMOS_DRIVE}"

Mounted at /content/drive


## 2. Código

In [4]:
if os.path.exists(REPO):
    !git -C {REPO} fetch -q origin {RAMA} && git -C {REPO} checkout -q {RAMA} && git -C {REPO} pull -q --ff-only
else:
    !git clone -q --branch {RAMA} https://github.com/gggelme/geo-project.git {REPO}
!git -C {REPO} log -1 --oneline

8f18ef3 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #19 from gggelme/baselines-2m


## 3. Datos al disco local

In [5]:
# Insumos chicos: de Drive al disco local (leerlos de Drive en cada ronda sería lento).
INSUMOS = "/content/insumos"
os.makedirs(INSUMOS, exist_ok=True)
for nombre in ["manifest.parquet", "splits.parquet", "validation.parquet"]:
    if not os.path.exists(f"{INSUMOS}/{nombre}"):
        shutil.copy(f"{INSUMOS_DRIVE}/{nombre}", f"{INSUMOS}/{nombre}")
print(os.listdir(INSUMOS))

['manifest.parquet', 'validation.parquet', 'splits.parquet']


In [6]:
# Baja cada parquet de train y val (no los de test) en la revisión fijada en la config, lo recomprime a 384 px y
# borra el original. ~30-60 min. Si se corta, volver a ejecutar: saltea los que ya están hechos.
import json
try:  # Token de HF desde los Secrets de Colab (el script corre en otro proceso y lo toma de la variable).
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("Sin HF_TOKEN en Secrets: se baja sin token (puede ser más lento).")
revision = json.load(open(f"{REPO}/{CONFIG}"))["hf_revision"]
procesos = max(1, (os.cpu_count() or 2) - 1)
!cd {REPO} && python scripts/recomprimir_shards.py --destino {SHARDS}/data --tmp /content/shards_tmp --splits {INSUMOS}/splits.parquet --revision {revision} --procesos {procesos}
total, usado, libre = shutil.disk_usage("/content")
print(f"Disco /content: {libre / 1e9:.0f} GB libres")

385 archivos -> /content/shards/data (384 px, calidad 90, 11 procesos)



data/train-00001-of-00393.parquet: downloading bytes:   0% 0.00/514M [00:00<?, ?B/s]
data/train-00007-of-00393.parquet: downloading bytes:   0% 0.00/502M [00:00<?, ?B/s]
data/train-00003-of-00393.parquet: reconstructing file:   0% 0.00/502M [00:00<?, ?B/s]
data/train-00005-of-00393.parquet: reconstructing file:   0% 0.00/503M [00:00<?, ?B/s]

data/train-00002-of-00393.parquet: downloading bytes:   0% 0.00/502M [00:00<?, ?B/s]
data/train-00009-of-00393.parquet: downloading bytes:   0% 0.00/503M [00:00<?, ?B/s]

data/train-00008-of-00393.parquet: downloading bytes:  42% 213M/503M [00:03<00:02, 111MB/s, 17.9MB/s  ]
data/train-00008-of-00393.parquet: downloading bytes:  46% 233M/503M [00:03<00:02, 127MB/s, 18.6MB/s  ]

data/train-00000-of-00393.parquet: downloading bytes:  74% 370M/501M [00:03<00:00, 200MB/s, 30.9MB/s  ]
data/train-00009-of-00393.parquet: downloading bytes:  57% 286M/503M [00:03<00:01, 144MB/s, 23.0M

## 4. Entrenamiento

Retoma solo si `CARPETA_CORRIDA` ya tiene checkpoints. La salida se ve en vivo; además todo queda en
`log.jsonl` e `historial.csv` dentro de la carpeta de la corrida.

In [7]:
import signal, sys
num_workers = max(2, (os.cpu_count() or 2) - 1)
comando = [sys.executable, "scripts/entrenar.py", "--config", CONFIG, "--cambiar",
           f"carpeta_corrida={CARPETA_CORRIDA}", f"datos_local={SHARDS}/data",
           f"manifest={INSUMOS}/manifest.parquet", f"splits={INSUMOS}/splits.parquet",
           f"val_global={INSUMOS}/validation.parquet", f"num_workers={num_workers}", f"max_horas={MAX_HORAS}"]
print(" ".join(comando))
proceso = subprocess.Popen(comando, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for linea in proceso.stdout:
        print(linea, end="")
except KeyboardInterrupt:
    # SIGTERM: el script termina el paso actual, guarda un checkpoint y sale (puede tardar si está evaluando).
    print("Interrumpido: se pide al script que guarde y salga (esperar, no interrumpir de nuevo)...")
    proceso.send_signal(signal.SIGTERM)
    while proceso.poll() is None:
        try:
            for linea in proceso.stdout:
                print(linea, end="")
        except KeyboardInterrupt:
            print("Sigue guardando; esperando a que salga...")
codigo = proceso.wait()
print({0: "TERMINADO", 3: "PAUSADO: volver a ejecutar todo para seguir"}.get(codigo, f"ERROR (código {codigo})"))

/usr/bin/python3 scripts/entrenar.py --config configs/baseline_paises_2m.json --cambiar carpeta_corrida=/content/drive/MyDrive/geo-project/entrenamiento_2m/baseline_paises_2m datos_local=/content/shards/data manifest=/content/insumos/manifest.parquet splits=/content/insumos/splits.parquet val_global=/content/insumos/validation.parquet num_workers=11 max_horas=20.0
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth

  0%|          | 0.00/44.7M [00:00<?, ?B/s]
 44%|████▎     | 19.5M/44.7M [00:00<00:00, 203MB/s]
 92%|█████████▏| 40.9M/44.7M [00:00<00:00, 215MB/s]
100%|██████████| 44.7M/44.7M [00:00<00:00, 214MB/s]
Corrida nueva.
tipo=inicio_sesion | sesion=1 | fase=base | dispositivo=cuda | train_imagenes=1920892 | train_panoramas=480246 | commit=8f18ef3f06d7dfee9512a48ec11dad4c4509cac0
  [base_1] paso 200: loss 4.1072, 288 img/s, espera de datos 64%, 6400/200000 muestras
  [base_1] paso 400: loss 3.9167, 43

## 5. Cómo va

In [8]:
import pandas as pd
ruta = f"{CARPETA_CORRIDA}/historial.csv"
if os.path.exists(ruta):
    h = pd.read_csv(ruta)
    display(h[["ronda", "loss_train", "val_ce_haversine", "val_top1", "val_top5", "val_km_mediana",
               "valglobal_top1", "valglobal_km_mediana", "lr_cabeza", "horas_acumuladas"]].tail(15))

,ronda,loss_train,val_ce_haversine,val_top1,val_top5,val_km_mediana,valglobal_top1,valglobal_km_mediana,lr_cabeza,horas_acumuladas
10,base_11,0.920650,4.091657,0.746363,0.957924,674.287676,0.653038,706.509601,0.00100,0.854740
11,base_12,0.881152,4.422441,0.687377,0.941801,709.467821,0.592273,759.370531,0.00100,0.932830
12,base_13,0.847114,4.432877,0.723948,0.955564,668.304356,0.641598,712.990357,0.00100,1.010205
13,base_14,0.811613,4.719686,0.702320,0.952222,679.331292,0.610278,732.570827,0.00100,1.087268
14,base_15,0.650979,5.011402,0.784506,0.967361,631.825817,0.701800,662.531537,0.00010,1.164346
15,base_16,0.566665,5.186141,0.775069,0.966575,634.706085,0.703676,665.201278,0.00010,1.242001
16,base_17,0.528753,5.639383,0.790012,0.967165,627.960635,0.711928,662.581500,0.00010,1.318913
17,base_18,0.503518,5.537585,0.793354,0.968738,629.826562,0.707989,667.072672,0.00010,1.395987
18,base_19,0.487964,5.455671,0.790012,0.967755,624.664570,0.705176,668.302254,0.00010,1.473866
19,base_20,0.465971,5.966265,0.796697,0.967755,625.090128,0.710053,665.201278,0.00010,1.550540
